# Solutions · 13 · Solid-liquid equilibria and alloy phase diagrams

Worked solutions to the exercises of [notebook 13](../notebooks/13_solid_liquid_equilibria_and_alloys.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engthermo is missing): install it from GitHub
    import engthermo
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engthermo"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from engthermo import sle
from engthermo.constants import R
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## Exercise 1

Tin (Tm 505 K, 7.03 kJ/mol) and lead (Tm 600.6 K, 4.77 kJ/mol) form the classic solder eutectic. Predict it with ideal liquids and compare with the experimental 183 degC at 0.74 mole fraction Sn. What does the discrepancy tell you about the Sn-Pb liquid?

In [2]:
e = sle.eutectic(505.0, 7.03e3, 600.6, 4.77e3)                        # component 1 = Sn, 2 = Pb
print(f"ideal-liquid prediction: eutectic at x_Sn = {e['x1']:.2f}, {e['T']-273.15:.0f} degC; experimental 0.74, 183 degC")

ideal-liquid prediction: eutectic at x_Sn = 0.49, 80 degC; experimental 0.74, 183 degC


Here the ideal model fails badly: it puts the eutectic a hundred kelvin too low and at the wrong composition.
Unlike Bi-Cd (notebook 13), the Sn-Pb liquid shows strong positive deviations from ideality (activity
coefficients above 1 raise both liquidus curves), and lead dissolves up to about 19 % tin in the solid, which
the immiscible-solids model cannot represent. The lesson cuts both ways: the same two numbers per metal
that predicted Bi-Cd within five kelvin are not enough for Sn-Pb. A real phase diagram needs the activity
coefficients of the melt - and a check against measurement.

## Exercise 2

Solar salt is 60/40 by mass NaNO3 (Tm 307 degC, 15.0 kJ/mol) / KNO3 (Tm 334 degC, 10.0 kJ/mol). Predict the eutectic of the ideal melt and compare with the observed 222 degC. Why does a low melting point matter for a thermal-storage medium?

In [3]:
e = sle.eutectic(307.0 + 273.15, 15.0e3, 334.0 + 273.15, 10.0e3)         # 1 = NaNO3, 2 = KNO3
M1, M2 = 84.99, 101.10
w1 = e["x1"] * M1 / (e["x1"] * M1 + (1 - e["x1"]) * M2)
print(f"ideal eutectic: x_NaNO3 = {e['x1']:.2f} ({w1:.0%} by mass), {e['T']-273.15:.0f} degC; observed about 222 degC (solar salt 60/40 by mass is near it)")

ideal eutectic: x_NaNO3 = 0.46 (42% by mass), 191 degC; observed about 222 degC (solar salt 60/40 by mass is near it)


The ideal-solution eutectic is about thirty degrees below the observed one - the two nitrates form solid
solutions and their melt is not ideal - but it lands in the right region and at nearly the right composition.
A low melting point matters because the salt must stay liquid in the tanks and pipes of a solar plant: every
degree of freezing point is a degree of heat tracing needed overnight, and the eutectic composition is the
safest choice against freezing.

## Exercise 3

**Implement it yourself:** compute the Cu-Ni liquidus and solidus at 1500 K from the two equilibrium constants by hand and compare with `sle.lens_diagram`.

In [4]:
T = 1500.0
K1, K2 = np.exp(13.05e3 / R * (1 / T - 1 / 1358.0)), np.exp(17.47e3 / R * (1 / T - 1 / 1728.0))
xL = (1 - K2) / (K1 - K2); xS = K1 * xL
lens = sle.lens_diagram(1358.0, 13.05e3, 1728.0, 17.47e3, n=401)
xL_lib, xS_lib = np.interp(T, lens["T"], lens["x1_liquidus"]), np.interp(T, lens["T"], lens["x1_solidus"])
print(f"by hand at 1500 K: liquidus x_Cu = {xL:.4f}, solidus x_Cu = {xS:.4f};  library (interpolated to 1500 K): {xL_lib:.4f}, {xS_lib:.4f}")

by hand at 1500 K: liquidus x_Cu = 0.6620, solidus x_Cu = 0.5934;  library (interpolated to 1500 K): 0.6620, 0.5934


Two equilibrium constants and two mole-fraction sums give both curves in closed form; the library does the
same at every temperature.